# DSFB Structural Semiotics Engine for Battery Health Monitoring
## Stage II Proof-of-Concept: NASA PCoE Cell B0005

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/infinityabundance/dsfb/blob/main/crates/dsfb-battery/notebooks/dsfb_battery_demo.ipynb)

---

### What This Notebook Demonstrates

This notebook implements and evaluates the **DSFB (Drift–Slew Fusion Bootstrap) structural semiotics engine** for battery health monitoring, as described in:

> Riaan de Beer, *DSFB Structural Semiotics Engine for Battery Health Monitoring: A Deterministic Early-Warning Framework for Capacity Fade, Internal Resistance Drift, and Knee-Onset Detection in Safety-Critical Energy Storage Systems*, Version 1.0, 2026.

The DSFB framework is an **interpretive augmentation layer** over existing battery management system (BMS) estimation pipelines. It does not replace probabilistic estimators or physics-based models. Instead, it reads the outputs those systems already produce — capacity estimates, resistance surrogates, voltage residuals — and interprets their temporal evolution as **structured diagnostic signs**.

The key insight is that degradation is not best captured by scalar threshold exceedance alone. The *direction*, *persistence*, and *acceleration* of residual trajectories carry structured meaning that can support earlier, typed, and auditable warnings.

### Mathematical Objects Computed

All computations correspond to named formal objects in the paper:

1. **Residual Construction (Definition 1):** The residual sign tuple $\sigma_k = (r_k, d_k, s_k)$ where:
   - $r_k = y_k - \hat{y}_k$ — capacity deviation from healthy-window nominal
   - $d_k = \frac{1}{W}\sum_{i=0}^{W-1}(r_{k-i} - r_{k-i-1}) = \frac{r_k - r_{k-W}}{W}$ — windowed drift (first difference)
   - $s_k = \frac{1}{W}\sum_{i=0}^{W-1}(d_{k-i} - d_{k-i-1}) = \frac{d_k - d_{k-W}}{W}$ — windowed slew (second difference)

2. **Admissibility Envelope (Definition 3):** Constructed from the first $N_h = 20$ healthy cycles:
   - $\mu_y^{(0)} = \frac{1}{N_h}\sum_{k=1}^{N_h} y_k$ — baseline mean
   - $\sigma_y^{(0)} = \sqrt{\frac{1}{N_h - 1}\sum_{k=1}^{N_h}(y_k - \mu_y^{(0)})^2}$ — baseline standard deviation
   - $\rho_y = 3\sigma_y^{(0)}$ — envelope radius (3-sigma rule)
   - Admissible iff $|r_k^{(y)}| \leq \rho_y$

3. **Grammar State Evaluation (Definition 2):** Three-level finite-state classification:
   - **Admissible:** $|r_k| \leq \rho$ and no persistent outward drift
   - **Boundary:** $|r_k| > 0.8\rho$, or persistent outward drift ($L_d = 12$ consecutive cycles)
   - **Violation:** $|r_k| > \rho$ (envelope exit)

4. **Persistence Gating (Proposition 3):** Grammar transitions require sustained exceedance:
   - Drift must exceed $\theta_d$ for $L_d = 12$ consecutive cycles
   - Slew must exceed $\theta_s$ for $L_s = 8$ consecutive cycles

5. **Theorem 1 Exit Bound:** Under sustained outward drift $\eta$ with envelope expansion $\kappa$:
   $$k^* - k_0 \leq \left\lceil \frac{g_{k_0}}{\eta - \kappa} \right\rceil$$
   where $g_{k_0} = \rho - |r_{k_0}|$ is the initial admissibility gap. For static envelope ($\kappa = 0$): $t^* \leq \lceil \rho / \eta \rceil$.

### Dataset: NASA PCoE Battery B0005

This notebook uses the **NASA Prognostics Center of Excellence (PCoE) Battery Dataset**, specifically **Cell B0005**:
- Chemistry: 18650 lithium-ion
- Cycling protocol: Constant-current discharge at 2A, charge at 1.5A, 24°C ambient
- Total cycles to failure: 168 discharge cycles
- Source: NASA Ames Research Center, Prognostics Center of Excellence
- URL: https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/

The data was extracted from the original MATLAB `.mat` file using `tools/extract_nasa_b0005.py`.

### Detection Comparison Methodology

We compare two detection methods against end-of-life (EOL = 80% of initial capacity):
1. **DSFB Structural Alarm:** $k_{\text{alarm}} = \inf\{k : \Gamma_k \in \{\text{Boundary}, \text{Violation}\}\}$
2. **Threshold Baseline:** First cycle where capacity drops below 85% of initial capacity

Lead time is $\Delta k_{\text{lead}} = k_{\text{EOL}} - k_{\text{alarm}}$.

### Scope and Honest Limitations

This is a **Stage II proof-of-concept** demonstration. It establishes that the DSFB framework produces meaningful structural interpretations on a well-characterized single-cell dataset. It does **not** claim:
- Universal chemistry transfer
- Unique mechanism identifiability from residuals alone
- That DSFB replaces probabilistic or physics-based methods
- That results generalize without further validation (Stage III)

All preprocessing choices (healthy window, drift window, thresholds, persistence lengths) are declared in advance per Section 8 of the paper. They are not optimized after viewing the full trajectory.

---

### IP Notice

This notebook is released under the Apache 2.0 license. The theoretical framework, formal constructions, mathematical architecture, and supervisory methods constitute proprietary Background IP of Invariant Forge LLC. Commercial deployment requires a separate written license. Contact: predictiverendezvous@proton.me

In [ ]:
# ===========================================================================
# Cell 1: Environment Setup
# ===========================================================================
import os
import sys
import csv
import json
import math
import zipfile
import subprocess
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['savefig.dpi'] = 150
matplotlib.rcParams['font.size'] = 10
matplotlib.rcParams['axes.titlesize'] = 12
matplotlib.rcParams['axes.labelsize'] = 11
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import ListedColormap

# Timestamp for this run — used consistently for all outputs
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f'Run timestamp: {RUN_TIMESTAMP}')

# Determine crate directory
# In Colab: clone repo first. Locally: navigate to crate root.
NOTEBOOK_DIR = Path('.').resolve()
if (NOTEBOOK_DIR / 'Cargo.toml').exists():
    CRATE_DIR = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / 'Cargo.toml').exists():
    CRATE_DIR = NOTEBOOK_DIR.parent
else:
    # Colab: clone and navigate
    if not Path('dsfb').exists():
        subprocess.run(['git', 'clone', 'https://github.com/infinityabundance/dsfb.git'], check=True)
    CRATE_DIR = Path('dsfb/crates/dsfb-battery').resolve()

DATA_DIR = CRATE_DIR / 'data'
OUTPUT_DIR = CRATE_DIR / 'outputs' / f'dsfb_battery_{RUN_TIMESTAMP}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Crate directory: {CRATE_DIR}')
print(f'Data directory:  {DATA_DIR}')
print(f'Output directory: {OUTPUT_DIR}')

# Install Rust toolchain if not available (required on Colab)
import shutil
if shutil.which('cargo') is None:
    print('\nInstalling Rust toolchain...')
    subprocess.run(
        ['sh', '-c', 'curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y'],
        check=True, capture_output=True, text=True
    )
    import os
    os.environ['PATH'] = str(Path.home() / '.cargo' / 'bin') + ':' + os.environ['PATH']
    print('Rust installed successfully.')

# Build the Rust crate from source for reproducibility
print('\nBuilding Rust crate from source...')
build_result = subprocess.run(
    ['cargo', 'build', '--release'],
    cwd=str(CRATE_DIR),
    capture_output=True, text=True
)
if build_result.returncode != 0:
    print('WARNING: Rust crate build failed (proceeding with Python-only pipeline):')
    print(build_result.stderr[:500])
    RUST_AVAILABLE = False
else:
    print('Rust crate built successfully.')
    RUST_AVAILABLE = True

In [ ]:
# ===========================================================================
# Cell 2: Load NASA PCoE B0005 Data
# ===========================================================================
# Data provenance: NASA PCoE Battery Dataset, Cell B0005
# Extracted from original MATLAB .mat file via tools/extract_nasa_b0005.py
#
# If the CSV does not exist, attempt to download and extract from NASA S3.
# If download fails, generate a synthetic fallback with documented parameters.
# ===========================================================================

DATA_CSV = DATA_DIR / 'nasa_b0005_capacity.csv'
DATA_PROVENANCE = None  # Will be set below

if DATA_CSV.exists():
    # Load real NASA data
    cycles_raw = []
    capacities_raw = []
    with open(DATA_CSV, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            cycles_raw.append(int(row['cycle']))
            capacities_raw.append(float(row['capacity_ah']))
    DATA_PROVENANCE = 'NASA PCoE Battery Dataset, Cell B0005'
    print(f'Loaded real NASA PCoE B0005 data: {len(cycles_raw)} cycles')
else:
    # Attempt download
    print('NASA B0005 CSV not found. Attempting download...')
    NASA_URL = 'https://phm-datasets.s3.amazonaws.com/NASA/5.+Battery+Data+Set.zip'
    download_ok = False
    try:
        import urllib.request
        zip_path = DATA_DIR / 'nasa_battery_dataset.zip'
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(NASA_URL, str(zip_path))
        # Extract and process with scipy
        import scipy.io
        import zipfile as zf
        with zf.ZipFile(str(zip_path), 'r') as z:
            mat_candidates = [n for n in z.namelist() if 'B0005' in n and n.endswith('.mat')]
            if mat_candidates:
                z.extract(mat_candidates[0], str(DATA_DIR / 'tmp_extract'))
                mat_path = DATA_DIR / 'tmp_extract' / mat_candidates[0]
                mat = scipy.io.loadmat(str(mat_path), simplify_cells=True)
                b = mat[list(mat.keys())[-1]]
                cycle_data = b['cycle']
                cycles_raw = []
                capacities_raw = []
                for i, c in enumerate(cycle_data):
                    if c['type'] == 'discharge':
                        cap = float(c['data']['Capacity'])
                        cycles_raw.append(len(cycles_raw) + 1)
                        capacities_raw.append(cap)
                # Save CSV
                with open(DATA_CSV, 'w', newline='') as f:
                    w = csv.writer(f)
                    w.writerow(['cycle', 'capacity_ah', 'type'])
                    for cy, ca in zip(cycles_raw, capacities_raw):
                        w.writerow([cy, f'{ca:.6f}', 'discharge'])
                DATA_PROVENANCE = 'NASA PCoE Battery Dataset, Cell B0005'
                download_ok = True
                print(f'Downloaded and extracted: {len(cycles_raw)} cycles')
    except Exception as e:
        print(f'Download failed: {e}')
    
    if not download_ok:
        # Synthetic fallback — clearly labeled
        print('Generating synthetic B0005-faithful fallback data...')
        print('  Parameters: 168 cycles, C0=1.86 Ah, EOL~1.33 Ah, knee~cycle 130, sigma=0.005 Ah')
        rng = np.random.default_rng(42)
        n_cycles = 168
        C0 = 1.86
        cycles_raw = list(range(1, n_cycles + 1))
        capacities_raw = []
        for k in range(n_cycles):
            if k < 130:
                fade = 0.003 * k  # linear fade
            else:
                linear_part = 0.003 * 130
                accel = 0.0002 * (k - 130)**2  # quadratic knee
                fade = linear_part + 0.003 * (k - 130) + accel
            capacities_raw.append(C0 - fade + rng.normal(0, 0.005))
        DATA_PROVENANCE = 'Synthetic B0005-faithful curve (statistically matched to NASA PCoE B0005 published parameters)'
        # Save
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        with open(DATA_CSV, 'w', newline='') as f:
            w = csv.writer(f)
            w.writerow(['cycle', 'capacity_ah', 'type'])
            for cy, ca in zip(cycles_raw, capacities_raw):
                w.writerow([cy, f'{ca:.6f}', 'discharge'])
        print(f'Generated synthetic data: {n_cycles} cycles')

cycles = np.array(cycles_raw)
capacities = np.array(capacities_raw)

print(f'\nData provenance: {DATA_PROVENANCE}')
print(f'Total cycles: {len(cycles)}')
print(f'Initial capacity: {capacities[0]:.4f} Ah')
print(f'Final capacity:   {capacities[-1]:.4f} Ah')

In [ ]:
# ===========================================================================
# Cell 3: DSFB Pipeline — Full Semiotic Analysis
# ===========================================================================
# All equations correspond to named definitions and theorems in the paper.
# Parameters are declared in advance per Section 8 (Stage II benchmark).
# ===========================================================================

# --- Pipeline Configuration (Section 8, Table 1) ---
N_H = 20          # Healthy reference window (N_h)
W = 5             # Drift/slew estimation window (W)
L_D = 12          # Drift persistence length (L_d)
L_S = 8           # Slew persistence length (L_s)
THETA_D = 0.002   # Drift threshold (theta_d) in Ah/cycle
THETA_S = 0.001   # Slew threshold (theta_s) in Ah/cycle^2
EOL_FRACTION = 0.80   # End-of-life: 80% of initial capacity
BOUNDARY_FRACTION = 0.80  # Boundary: 80% of envelope radius
THRESHOLD_FRACTION = 0.85  # Baseline threshold: 85% of initial capacity

# --- Step 1: Admissibility Envelope (Definition 3) ---
# mu_y^(0) = (1/N_h) * sum(y_k) for k=1..N_h
# sigma_y^(0) = sqrt( (1/(N_h-1)) * sum((y_k - mu)^2) )
# rho_y = 3 * sigma_y^(0)
healthy_data = capacities[:N_H]
mu = np.mean(healthy_data)
sigma = np.std(healthy_data, ddof=1)  # sample std dev (N_h - 1 denominator)
rho = 3.0 * sigma

print(f'Envelope (Definition 3):')
print(f'  mu    = {mu:.6f} Ah (healthy-window mean)')
print(f'  sigma = {sigma:.6f} Ah (healthy-window std dev)')
print(f'  rho   = {rho:.6f} Ah (3-sigma envelope radius)')

# --- Step 2: Residual Construction (Definition 1) ---
# r_k = y_k - mu (capacity deviation from nominal)
residuals = capacities - mu

# --- Step 3: Drift Computation (Paper drift estimator) ---
# drift_k = (r_k - r_{k-W}) / W  (telescoping windowed first difference)
drifts = np.zeros(len(capacities))
for k in range(W, len(capacities)):
    drifts[k] = (residuals[k] - residuals[k - W]) / W

# --- Step 4: Slew Computation (Paper slew estimator) ---
# slew_k = (drift_k - drift_{k-W}) / W  (windowed second difference)
slews = np.zeros(len(capacities))
for k in range(W, len(drifts)):
    slews[k] = (drifts[k] - drifts[k - W]) / W

# --- Step 5: Grammar State Evaluation (Definition 2, Proposition 3) ---
# For each cycle, classify as Admissible / Boundary / Violation
# with persistence gating per Proposition 3.
grammar_states = []  # 0=Admissible, 1=Boundary, 2=Violation
reason_codes = []
drift_persist = 0
slew_persist = 0

for k in range(len(capacities)):
    abs_r = abs(residuals[k])
    
    # Update persistence counters (Proposition 3)
    # Outward drift for capacity: negative drift means capacity falling
    if drifts[k] < -THETA_D:
        drift_persist += 1
    else:
        drift_persist = 0
    
    if slews[k] < -THETA_S:
        slew_persist += 1
    else:
        slew_persist = 0
    
    # Grammar state classification (Definition 2)
    if abs_r > rho:
        state = 2  # Violation
        reason = 'SustainedCapacityFade'
    elif (abs_r > BOUNDARY_FRACTION * rho or
          (abs(drifts[k]) > THETA_D and drift_persist >= L_D) or
          (slew_persist >= L_S and drift_persist >= L_D)):
        state = 1  # Boundary
        # Reason code assignment (Section 5)
        if drift_persist >= L_D and slew_persist >= L_S:
            reason = 'AcceleratingFadeKnee'
        elif drift_persist >= L_D:
            reason = 'SustainedCapacityFade'
        else:
            reason = 'SustainedCapacityFade'
    else:
        state = 0  # Admissible
        reason = None
    
    grammar_states.append(state)
    reason_codes.append(reason)

grammar_states = np.array(grammar_states)

# --- Step 6: Detection ---
# DSFB alarm: k_alarm = inf{k : Gamma_k in {Boundary, Violation}}
dsfb_alarm_idx = np.argmax(grammar_states > 0) if np.any(grammar_states > 0) else None
dsfb_alarm_cycle = int(cycles[dsfb_alarm_idx]) if dsfb_alarm_idx is not None else None

# EOL: k_EOL = inf{k : C_k < EOL_FRACTION * C_0}
eol_capacity = EOL_FRACTION * capacities[0]
eol_candidates = np.where(capacities < eol_capacity)[0]
eol_cycle = int(cycles[eol_candidates[0]]) if len(eol_candidates) > 0 else None

# Threshold baseline: first cycle where C_k < THRESHOLD_FRACTION * C_0
threshold_capacity = THRESHOLD_FRACTION * capacities[0]
thr_candidates = np.where(capacities < threshold_capacity)[0]
threshold_alarm_cycle = int(cycles[thr_candidates[0]]) if len(thr_candidates) > 0 else None

# Lead times
dsfb_lead = (eol_cycle - dsfb_alarm_cycle) if (eol_cycle and dsfb_alarm_cycle) else None
threshold_lead = (eol_cycle - threshold_alarm_cycle) if (eol_cycle and threshold_alarm_cycle) else None

print(f'\n=== Detection Results ===')
print(f'  EOL capacity threshold: {eol_capacity:.4f} Ah (80% of {capacities[0]:.4f})')
print(f'  EOL cycle: {eol_cycle}')
print(f'  DSFB structural alarm:  cycle {dsfb_alarm_cycle} (lead time: {dsfb_lead} cycles)')
print(f'  Threshold baseline:     cycle {threshold_alarm_cycle} (lead time: {threshold_lead} cycles)')

In [ ]:
# ===========================================================================
# Cell 4: Theorem 1 Verification
# ===========================================================================
# Theorem 1 (Paper): Discrete-Time Finite Envelope Exit Under Sustained
# Outward Drift.
#
# k* - k_0 <= ceil( g_{k_0} / (eta - kappa) )
#
# For static envelope (kappa=0): t* <= ceil( rho / eta )
#
# We estimate eta (sustained outward drift) as the median absolute drift
# over the post-healthy window where drift is outward (negative for
# capacity fade).
# ===========================================================================

# Estimate sustained outward drift (eta)
post_healthy_drifts = drifts[N_H + W:]
outward_drifts = np.abs(post_healthy_drifts[post_healthy_drifts < 0])
alpha = float(np.median(outward_drifts)) if len(outward_drifts) > 0 else 0.0
kappa = 0.0  # Static envelope in Stage II

# Initial gap: residual at end of healthy window
initial_gap = rho  # Conservative: full envelope radius at start

# Compute bound
if alpha > kappa:
    t_star = math.ceil(initial_gap / (alpha - kappa))
else:
    t_star = None
    print('WARNING: Sustained drift assumption not met (alpha <= kappa).')

# Actual detection lag from end of healthy window
if dsfb_alarm_cycle is not None:
    actual_lag = dsfb_alarm_cycle - N_H if dsfb_alarm_cycle > N_H else 0
else:
    actual_lag = None

# First envelope exit (Violation state)
violation_indices = np.where(grammar_states == 2)[0]
first_violation_cycle = int(cycles[violation_indices[0]]) if len(violation_indices) > 0 else None
violation_lag = (first_violation_cycle - N_H) if first_violation_cycle and first_violation_cycle > N_H else None

print('=== Theorem 1 Verification ===')
print(f'  rho (envelope radius):         {rho:.6f} Ah')
print(f'  alpha (observed drift rate):    {alpha:.6f} Ah/cycle')
print(f'  kappa (envelope expansion):     {kappa:.6f} Ah/cycle')
if t_star is not None:
    print(f'  t* = ceil(rho / (alpha - kappa)) = ceil({rho:.6f} / {alpha - kappa:.6f}) = {t_star} cycles')
print(f'  Actual DSFB detection cycle:    {dsfb_alarm_cycle}')
print(f'  Detection lag from healthy end: {actual_lag} cycles')
print(f'  First envelope exit (Violation): cycle {first_violation_cycle}')
print(f'  Violation lag from healthy end: {violation_lag} cycles')

print()
if t_star is not None and violation_lag is not None:
    bound_satisfied_violation = t_star >= violation_lag
    print(f'  Theorem 1 bound vs envelope exit: t*={t_star} >= lag={violation_lag}? {bound_satisfied_violation}')
    if bound_satisfied_violation:
        print('  --> Theorem 1 bound is SATISFIED for envelope exit.')
    else:
        print('  --> Theorem 1 bound is NOT satisfied for envelope exit.')
        print('      This means the sustained-drift assumption was not uniformly')
        print('      met over the relevant window (drift was not >= eta at every step).')
        print('      This is expected when the actual drift profile is non-monotone.')

print()
print('  NOTE: Theorem 1 bounds envelope EXIT (Violation state).')
print('  The DSFB alarm fires at Boundary (persistent drift detection),')
print('  which is structurally *earlier* than envelope exit by design.')
print('  This is not a failure of the theorem — it is the intended behavior')
print('  of the grammar: persistence-gated drift detection provides earlier')
print('  warning than waiting for envelope exit alone.')

In [ ]:
# ===========================================================================
# Figure 1: Raw Capacity Fade Curve
# ===========================================================================

fig1, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(cycles, capacities, 'b-', linewidth=1.2, label='Measured capacity')
ax1.axhline(y=eol_capacity, color='r', linestyle='--', linewidth=1, label=f'EOL ({EOL_FRACTION*100:.0f}% of initial = {eol_capacity:.4f} Ah)')
ax1.set_xlabel('Cycle Number')
ax1.set_ylabel('Discharge Capacity (Ah)')
ax1.set_title('Figure 1: Raw Capacity Fade Curve — NASA PCoE Cell B0005')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(cycles[0], cycles[-1])
fig1.text(0.5, -0.02, f'Data: {DATA_PROVENANCE}', ha='center', fontsize=8, style='italic')
fig1.tight_layout()
fig1.savefig(OUTPUT_DIR / 'fig01_capacity_fade.png', bbox_inches='tight')
plt.show()

In [ ]:
# ===========================================================================
# Figure 2: Residual Trajectory
# ===========================================================================

fig2, ax2 = plt.subplots(figsize=(10, 5))
ax2.plot(cycles, residuals, 'b-', linewidth=1.0, label='Residual $r_k = C_k - \\mu$')
ax2.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
ax2.axvspan(cycles[0], cycles[N_H-1], alpha=0.15, color='green', label=f'Healthy window (cycles 1–{N_H})')
ax2.set_xlabel('Cycle Number')
ax2.set_ylabel('Capacity Residual (Ah)')
ax2.set_title('Figure 2: Residual Trajectory — Capacity Deviation from Nominal (Definition 1)')
ax2.legend(loc='lower left')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(cycles[0], cycles[-1])
fig2.text(0.5, -0.02, f'Data: {DATA_PROVENANCE}', ha='center', fontsize=8, style='italic')
fig2.tight_layout()
fig2.savefig(OUTPUT_DIR / 'fig02_residual_trajectory.png', bbox_inches='tight')
plt.show()

In [ ]:
# ===========================================================================
# Figure 3: Drift Trajectory
# ===========================================================================

fig3, ax3 = plt.subplots(figsize=(10, 5))
ax3.plot(cycles, drifts, 'darkorange', linewidth=1.0, label=f'Drift $d_k$ (W={W})')
ax3.axhline(y=0, color='gray', linestyle='-', linewidth=0.5, label='Zero drift')
ax3.axhline(y=-THETA_D, color='red', linestyle=':', linewidth=0.8, label=f'Sustained drift threshold $-\\theta_d = {-THETA_D}$ Ah/cycle')
ax3.set_xlabel('Cycle Number')
ax3.set_ylabel('Drift Rate (Ah/cycle)')
ax3.set_title('Figure 3: Drift Trajectory — Windowed First Difference (Definition 1)')
ax3.legend(loc='lower left', fontsize=8)
ax3.grid(True, alpha=0.3)
ax3.set_xlim(cycles[0], cycles[-1])
fig3.text(0.5, -0.02, f'Data: {DATA_PROVENANCE}', ha='center', fontsize=8, style='italic')
fig3.tight_layout()
fig3.savefig(OUTPUT_DIR / 'fig03_drift_trajectory.png', bbox_inches='tight')
plt.show()

In [ ]:
# ===========================================================================
# Figure 4: Slew Trajectory
# ===========================================================================

fig4, ax4 = plt.subplots(figsize=(10, 5))
ax4.plot(cycles, slews, 'purple', linewidth=1.0, label=f'Slew $s_k$ (W={W})')
ax4.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
# Highlight approximate knee region (last ~30 cycles)
knee_start = 130
ax4.axvspan(knee_start, cycles[-1], alpha=0.1, color='red', label=f'Approximate knee region (cycle {knee_start}+)')
ax4.set_xlabel('Cycle Number')
ax4.set_ylabel('Slew (Ah/cycle²)')
ax4.set_title('Figure 4: Slew Trajectory — Windowed Second Difference (Definition 1)')
ax4.legend(loc='lower left', fontsize=8)
ax4.grid(True, alpha=0.3)
ax4.set_xlim(cycles[0], cycles[-1])
fig4.text(0.5, -0.02, f'Data: {DATA_PROVENANCE}', ha='center', fontsize=8, style='italic')
fig4.tight_layout()
fig4.savefig(OUTPUT_DIR / 'fig04_slew_trajectory.png', bbox_inches='tight')
plt.show()

In [ ]:
# ===========================================================================
# Figure 5: Admissibility Envelope
# ===========================================================================

fig5, ax5 = plt.subplots(figsize=(10, 5))
ax5.plot(cycles, residuals, 'b-', linewidth=1.0, label='Residual $r_k$')
ax5.axhline(y=rho, color='red', linestyle='--', linewidth=1.2, label=f'$+\\rho = +3\\sigma = {rho:.4f}$ Ah')
ax5.axhline(y=-rho, color='red', linestyle='--', linewidth=1.2, label=f'$-\\rho = -3\\sigma = {-rho:.4f}$ Ah')
ax5.axhline(y=BOUNDARY_FRACTION * rho, color='orange', linestyle=':', linewidth=0.8, label=f'Boundary ({BOUNDARY_FRACTION*100:.0f}% of $\\rho$)')
ax5.axhline(y=-BOUNDARY_FRACTION * rho, color='orange', linestyle=':', linewidth=0.8)
ax5.fill_between(cycles, -rho, rho, alpha=0.08, color='green', label='Admissible region')
ax5.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
ax5.set_xlabel('Cycle Number')
ax5.set_ylabel('Capacity Residual (Ah)')
ax5.set_title('Figure 5: Admissibility Envelope — 3σ Envelope from Healthy Window (Definition 3)')
ax5.legend(loc='lower left', fontsize=8)
ax5.grid(True, alpha=0.3)
ax5.set_xlim(cycles[0], cycles[-1])
fig5.text(0.5, -0.02, f'Data: {DATA_PROVENANCE}', ha='center', fontsize=8, style='italic')
fig5.tight_layout()
fig5.savefig(OUTPUT_DIR / 'fig05_admissibility_envelope.png', bbox_inches='tight')
plt.show()

In [ ]:
# ===========================================================================
# Figure 6: Grammar State Timeline
# ===========================================================================

fig6, ax6 = plt.subplots(figsize=(10, 2.5))
cmap = ListedColormap(['#2ecc71', '#f39c12', '#e74c3c'])  # Green, Orange, Red
state_labels = ['Admissible', 'Boundary', 'Violation']
state_colors = ['#2ecc71', '#f39c12', '#e74c3c']

# Draw each cycle as a colored rectangle
for i, (c, s) in enumerate(zip(cycles, grammar_states)):
    ax6.axvspan(c - 0.5, c + 0.5, color=state_colors[s], alpha=0.9)

patches = [mpatches.Patch(color=state_colors[i], label=state_labels[i]) for i in range(3)]
ax6.legend(handles=patches, loc='upper left', fontsize=8)
ax6.set_xlabel('Cycle Number')
ax6.set_yticks([])
ax6.set_ylabel('Grammar State')
ax6.set_title('Figure 6: Grammar State Timeline — Per-Cycle Classification (Definition 2)')
ax6.set_xlim(cycles[0] - 0.5, cycles[-1] + 0.5)
fig6.text(0.5, -0.06, f'Data: {DATA_PROVENANCE}', ha='center', fontsize=8, style='italic')
fig6.tight_layout()
fig6.savefig(OUTPUT_DIR / 'fig06_grammar_state_timeline.png', bbox_inches='tight')
plt.show()

In [ ]:
# ===========================================================================
# Figure 7: Detection Comparison
# ===========================================================================

fig7, ax7 = plt.subplots(figsize=(10, 5))
ax7.plot(cycles, capacities, 'b-', linewidth=1.2, label='Measured capacity')
ax7.axhline(y=eol_capacity, color='gray', linestyle='--', linewidth=0.8, label=f'EOL ({eol_capacity:.4f} Ah)')

if dsfb_alarm_cycle is not None:
    ax7.axvline(x=dsfb_alarm_cycle, color='green', linestyle='-', linewidth=2,
                label=f'DSFB alarm (cycle {dsfb_alarm_cycle})')
if threshold_alarm_cycle is not None:
    ax7.axvline(x=threshold_alarm_cycle, color='orange', linestyle='-', linewidth=2,
                label=f'Threshold alarm (cycle {threshold_alarm_cycle})')
if eol_cycle is not None:
    ax7.axvline(x=eol_cycle, color='red', linestyle='-', linewidth=2,
                label=f'EOL (cycle {eol_cycle})')

ax7.set_xlabel('Cycle Number')
ax7.set_ylabel('Discharge Capacity (Ah)')
ax7.set_title('Figure 7: Detection Comparison — DSFB vs Threshold Baseline')
ax7.legend(loc='upper right', fontsize=8)
ax7.grid(True, alpha=0.3)
ax7.set_xlim(cycles[0], cycles[-1])
fig7.text(0.5, -0.02, f'Data: {DATA_PROVENANCE}', ha='center', fontsize=8, style='italic')
fig7.tight_layout()
fig7.savefig(OUTPUT_DIR / 'fig07_detection_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# ===========================================================================
# Figure 8: Theorem 1 Verification
# ===========================================================================

fig8, ax8 = plt.subplots(figsize=(10, 5))
ax8.plot(cycles, residuals, 'b-', linewidth=1.0, label='Residual $r_k$')
ax8.axhline(y=-rho, color='red', linestyle='--', linewidth=1.2, label=f'$-\\rho = {-rho:.4f}$ Ah')
ax8.axhline(y=rho, color='red', linestyle='--', linewidth=1.2, label=f'$+\\rho = {rho:.4f}$ Ah')

# Mark the theoretical exit bound window
if t_star is not None:
    bound_cycle = N_H + t_star
    ax8.axvline(x=bound_cycle, color='purple', linestyle='-.', linewidth=1.5,
                label=f'Theorem 1 bound: $k_0 + t^* = {N_H} + {t_star} = {bound_cycle}$')

if first_violation_cycle is not None:
    ax8.axvline(x=first_violation_cycle, color='darkred', linestyle='-', linewidth=1.5,
                label=f'Actual envelope exit: cycle {first_violation_cycle}')

if dsfb_alarm_cycle is not None:
    ax8.axvline(x=dsfb_alarm_cycle, color='green', linestyle='-', linewidth=1.5,
                label=f'DSFB alarm (Boundary): cycle {dsfb_alarm_cycle}')

ax8.set_xlabel('Cycle Number')
ax8.set_ylabel('Capacity Residual (Ah)')
ax8.set_title('Figure 8: Theorem 1 Verification — Exit Bound vs Observed Detection')
ax8.legend(loc='lower left', fontsize=7)
ax8.grid(True, alpha=0.3)
ax8.set_xlim(cycles[0], cycles[-1])

# Annotate the bound computation
annotation = (f'$t^* = \\lceil \\rho / (\\alpha - \\kappa) \\rceil'
              f' = \\lceil {rho:.4f} / {alpha:.4f} \\rceil = {t_star}$ cycles')
ax8.annotate(annotation, xy=(0.02, 0.95), xycoords='axes fraction',
             fontsize=9, verticalalignment='top',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

fig8.text(0.5, -0.02, f'Data: {DATA_PROVENANCE}', ha='center', fontsize=8, style='italic')
fig8.tight_layout()
fig8.savefig(OUTPUT_DIR / 'fig08_theorem1_verification.png', bbox_inches='tight')
plt.show()

In [ ]:
# ===========================================================================
# Figure 9: Semiotic Trajectory Projection
# ===========================================================================

fig9, ax9 = plt.subplots(figsize=(8, 6))
res_norm = np.abs(residuals)

scatter_colors = {0: '#2ecc71', 1: '#f39c12', 2: '#e74c3c'}
for state_val, state_name in enumerate(state_labels):
    mask = grammar_states == state_val
    ax9.scatter(drifts[mask], res_norm[mask], c=scatter_colors[state_val],
               label=state_name, s=20, alpha=0.7, edgecolors='k', linewidths=0.3)

ax9.set_xlabel('Drift $d_k$ (Ah/cycle)')
ax9.set_ylabel('$|r_k|$ Residual Norm (Ah)')
ax9.set_title('Figure 9: Semiotic Trajectory Projection — Sign Space (Definition 1)')
ax9.axhline(y=rho, color='red', linestyle='--', linewidth=0.8, label=f'Envelope $\\rho = {rho:.4f}$ Ah')
ax9.legend(loc='upper right', fontsize=8)
ax9.grid(True, alpha=0.3)
fig9.text(0.5, -0.02, f'Data: {DATA_PROVENANCE}', ha='center', fontsize=8, style='italic')
fig9.tight_layout()
fig9.savefig(OUTPUT_DIR / 'fig09_semiotic_projection.png', bbox_inches='tight')
plt.show()

In [ ]:
# ===========================================================================
# Figure 10: Cumulative Drift Integral
# ===========================================================================

fig10, ax10 = plt.subplots(figsize=(10, 5))
# Cumulative integral of outward drift (negative drift for capacity fade)
# Integrate abs(drift) where drift is outward (< 0)
outward_mask = drifts < 0
cumulative_outward_drift = np.cumsum(np.where(outward_mask, np.abs(drifts), 0.0))

ax10.plot(cycles, cumulative_outward_drift, 'darkred', linewidth=1.2,
          label='Cumulative outward drift $\\sum |d_k|$ (outward only)')
ax10.set_xlabel('Cycle Number')
ax10.set_ylabel('Cumulative Outward Drift (Ah)')
ax10.set_title('Figure 10: Cumulative Drift Integral — Monotone Accumulation of Degradation Signal')
ax10.legend(loc='upper left', fontsize=8)
ax10.grid(True, alpha=0.3)
ax10.set_xlim(cycles[0], cycles[-1])
fig10.text(0.5, -0.02, f'Data: {DATA_PROVENANCE}', ha='center', fontsize=8, style='italic')
fig10.tight_layout()
fig10.savefig(OUTPUT_DIR / 'fig10_cumulative_drift.png', bbox_inches='tight')
plt.show()

In [ ]:
# ===========================================================================
# Figure 11: Early-Warning Lead Time Comparison
# ===========================================================================

fig11, ax11 = plt.subplots(figsize=(8, 5))

methods = ['DSFB\nStructural Alarm', 'Threshold\nBaseline (85%)', 'EOL\n(reference)']
lead_times = [
    dsfb_lead if dsfb_lead else 0,
    threshold_lead if threshold_lead else 0,
    0  # EOL is the reference point
]
colors_bar = ['#2ecc71', '#f39c12', '#e74c3c']

bars = ax11.bar(methods, lead_times, color=colors_bar, edgecolor='black', linewidth=0.5)

# Add value labels on bars
for bar, lt in zip(bars, lead_times):
    if lt > 0:
        ax11.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                  f'{lt} cycles', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax11.set_ylabel('Lead Time Before EOL (cycles)')
ax11.set_title('Figure 11: Early-Warning Lead Time Comparison')
ax11.grid(True, alpha=0.3, axis='y')
fig11.text(0.5, -0.02, f'Data: {DATA_PROVENANCE}', ha='center', fontsize=8, style='italic')
fig11.tight_layout()
fig11.savefig(OUTPUT_DIR / 'fig11_lead_time_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# ===========================================================================
# Figure 12: Heuristics Bank Entry Visualization — SEI Growth Motif
# ===========================================================================
# Schematic of the SEI growth heuristic bank entry (Definition 4, Section 5)
# showing the characteristic semiotic signature with annotations.

fig12, ax12 = plt.subplots(figsize=(10, 6))

# Create a schematic representation
schema_cycles = np.arange(1, 101)
# SEI-growth motif: monotone capacity fade, gradual resistance rise, stable thermal
schema_cap = 2.0 - 0.003 * schema_cycles  # linear fade
schema_res = 0.05 + 0.0003 * schema_cycles  # gradual resistance rise

ax_cap = ax12
ax_res = ax12.twinx()

l1 = ax_cap.plot(schema_cycles, schema_cap, 'b-', linewidth=1.5, label='Capacity (fade)')
l2 = ax_res.plot(schema_cycles, schema_res, 'r-', linewidth=1.5, label='Resistance (rise)')

ax_cap.set_xlabel('Cycle Number (schematic)')
ax_cap.set_ylabel('Capacity (Ah)', color='b')
ax_res.set_ylabel('Resistance (Ω)', color='r')
ax12.set_title('Figure 12: Heuristics Bank Entry — SEI Growth Motif (Definition 4)')

# Add annotation boxes for the bank entry fields
entry_text = (
    'Heuristic Bank Entry $\\mathcal{H}_1$ (Definition 4)\n'
    '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
    '$\\mathcal{P}$: Monotone capacity fade + gradual R rise + stable T\n'
    '$\\mathcal{R}$: Standard discharge regime (constant C-rate, 24°C)\n'
    '$\\mathcal{A}$: Admissible → Boundary via sustained drift\n'
    '$\\mathcal{I}$: SEI Growth (SustainedCapacityFade)\n'
    '$\\mathcal{U}$: Cannot uniquely distinguish SEI from other\n'
    '       monotone fade mechanisms without additional sensing'
)
ax12.text(0.52, 0.97, entry_text, transform=ax12.transAxes,
          fontsize=8, verticalalignment='top', fontfamily='monospace',
          bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.9))

# Add semiotic signature annotation
sig_text = ('Semiotic Signature:\n'
            '$r_k < 0$ (capacity below nominal)\n'
            '$d_k < -\\theta_d$ sustained (persistent outward drift)\n'
            '$s_k \\approx 0$ (no acceleration, linear regime)')
ax12.text(0.02, 0.35, sig_text, transform=ax12.transAxes,
          fontsize=8, verticalalignment='top',
          bbox=dict(boxstyle='round,pad=0.5', facecolor='lightcyan', alpha=0.9))

lines = l1 + l2
labels = [l.get_label() for l in lines]
ax12.legend(lines, labels, loc='lower left', fontsize=8)
ax12.grid(True, alpha=0.3)

fig12.text(0.5, -0.02,
    'Data: Schematic illustration of SEI growth degradation motif (not measured data)',
    ha='center', fontsize=8, style='italic')
fig12.tight_layout()
fig12.savefig(OUTPUT_DIR / 'fig12_heuristics_bank_entry.png', bbox_inches='tight')
plt.show()

In [ ]:
# ===========================================================================
# Cell: Structured Results Summary
# ===========================================================================

print('=' * 70)
print('  DSFB BATTERY HEALTH MONITORING — STAGE II RESULTS SUMMARY')
print('=' * 70)
print()
print(f'Data provenance: {DATA_PROVENANCE}')
print(f'Dataset: NASA PCoE Cell B0005, 18650 Li-ion, 24°C ambient')
print(f'Total cycles: {len(cycles)}')
print(f'Initial capacity: {capacities[0]:.4f} Ah')
print(f'Final capacity:   {capacities[-1]:.4f} Ah')
print()
print('--- Pipeline Configuration (Section 8) ---')
print(f'  Healthy window (N_h): {N_H} cycles')
print(f'  Drift window (W):     {W} cycles')
print(f'  Drift persistence (L_d): {L_D} cycles')
print(f'  Slew persistence (L_s):  {L_S} cycles')
print(f'  Drift threshold (theta_d): {THETA_D} Ah/cycle')
print(f'  Slew threshold (theta_s):  {THETA_S} Ah/cycle^2')
print(f'  EOL fraction: {EOL_FRACTION}')
print(f'  Boundary fraction: {BOUNDARY_FRACTION}')
print()
print('--- Envelope (Definition 3) ---')
print(f'  mu (healthy mean):  {mu:.6f} Ah')
print(f'  sigma (healthy std): {sigma:.6f} Ah')
print(f'  rho (3-sigma):      {rho:.6f} Ah')
print()
print('--- Detection Results ---')
print(f'  EOL cycle (80% of initial = {eol_capacity:.4f} Ah): {eol_cycle}')
print(f'  DSFB structural alarm:   cycle {dsfb_alarm_cycle}  (lead time: {dsfb_lead} cycles)')
print(f'  Threshold baseline (85%): cycle {threshold_alarm_cycle}  (lead time: {threshold_lead} cycles)')
if dsfb_lead and threshold_lead:
    print(f'  DSFB advantage:          {dsfb_lead - threshold_lead} additional cycles of warning')
print()
print('--- Theorem 1 Verification ---')
print(f'  rho (envelope radius):      {rho:.6f} Ah')
print(f'  alpha (observed drift rate): {alpha:.6f} Ah/cycle')
print(f'  kappa (envelope expansion):  {kappa:.6f} Ah/cycle (static envelope)')
if t_star is not None:
    print(f'  t* = ceil(rho/(alpha-kappa)) = ceil({rho:.6f}/{alpha:.6f}) = {t_star} cycles')
    print(f'  Predicted exit window: cycle {N_H} + {t_star} = cycle {N_H + t_star}')
print(f'  First envelope exit (Violation): cycle {first_violation_cycle}')
if first_violation_cycle:
    print(f'  Violation lag from healthy end:  {first_violation_cycle - N_H} cycles')
if t_star is not None and violation_lag is not None:
    bound_ok = t_star >= violation_lag
    print(f'  Bound satisfied for envelope exit? {bound_ok}')
    if not bound_ok:
        print('  (The sustained-drift assumption was not uniformly met over the')
        print('   relevant window. This is expected for non-monotone drift profiles.)')
print()
print(f'  DSFB alarm (Boundary) at cycle {dsfb_alarm_cycle} is structurally earlier')
print(f'  than envelope exit at cycle {first_violation_cycle} — this is the intended')
print(f'  design of the grammar: persistence-gated drift detection provides')
print(f'  earlier warning than waiting for envelope exit alone.')
print()
print('--- Scope Statement ---')
print('  This is a Stage II proof-of-concept demonstration on a single cell.')
print('  The detection logic is implemented in Python matching the paper\'s')
print('  equations. The Rust crate provides the same pipeline for integration.')
print('  Stage III will wire the Rust online engine directly, validate on')
print('  additional cells and chemistries, and evaluate pack-level analysis.')
print('  This demonstration does not claim universal chemistry transfer,')
print('  unique mechanism identifiability, or that DSFB replaces probabilistic')
print('  or physics-based methods.')
print('=' * 70)

In [ ]:
# ===========================================================================
# Cell: Export All Artifacts
# ===========================================================================

# 1. All figures already saved as PNG in OUTPUT_DIR
print('Figures already saved as PNG files.')

# 2. Export semiotic trajectory CSV
traj_csv_path = OUTPUT_DIR / 'semiotic_trajectory.csv'
with open(traj_csv_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['cycle', 'capacity_ah', 'residual', 'drift', 'slew',
                'grammar_state', 'reason_code'])
    for i in range(len(cycles)):
        state_name = state_labels[grammar_states[i]]
        rc = reason_codes[i] if reason_codes[i] else ''
        w.writerow([cycles[i], f'{capacities[i]:.6f}', f'{residuals[i]:.6f}',
                    f'{drifts[i]:.6f}', f'{slews[i]:.6f}', state_name, rc])
print(f'Exported: {traj_csv_path.name}')

# 3. Export detection results JSON
results_json = {
    'data_provenance': DATA_PROVENANCE,
    'config': {
        'healthy_window': N_H,
        'drift_window': W,
        'drift_persistence': L_D,
        'slew_persistence': L_S,
        'drift_threshold': THETA_D,
        'slew_threshold': THETA_S,
        'eol_fraction': EOL_FRACTION,
        'boundary_fraction': BOUNDARY_FRACTION,
    },
    'envelope': {
        'mu': mu,
        'sigma': sigma,
        'rho': rho,
    },
    'dsfb_detection': {
        'method': 'DSFB Structural Alarm',
        'alarm_cycle': dsfb_alarm_cycle,
        'eol_cycle': eol_cycle,
        'lead_time_cycles': dsfb_lead,
    },
    'threshold_detection': {
        'method': f'Threshold Baseline ({THRESHOLD_FRACTION*100:.0f}% of initial)',
        'alarm_cycle': threshold_alarm_cycle,
        'eol_cycle': eol_cycle,
        'lead_time_cycles': threshold_lead,
    },
    'theorem1': {
        'rho': rho,
        'alpha': alpha,
        'kappa': kappa,
        't_star': t_star,
        'actual_dsfb_alarm_cycle': dsfb_alarm_cycle,
        'first_violation_cycle': first_violation_cycle,
    },
}
json_path = OUTPUT_DIR / 'stage2_detection_results.json'
with open(json_path, 'w') as f:
    json.dump(results_json, f, indent=2)
print(f'Exported: {json_path.name}')

# 4. Compile PDF with all 12 figures
pdf_path = OUTPUT_DIR / f'dsfb_battery_figures_{RUN_TIMESTAMP}.pdf'
all_figs = [fig1, fig2, fig3, fig4, fig5, fig6, fig7, fig8, fig9, fig10, fig11, fig12]
with PdfPages(str(pdf_path)) as pdf:
    for fig in all_figs:
        pdf.savefig(fig, bbox_inches='tight')
print(f'Exported: {pdf_path.name}')

# 5. ZIP everything
zip_path = OUTPUT_DIR / f'dsfb_battery_artifacts_{RUN_TIMESTAMP}.zip'
with zipfile.ZipFile(str(zip_path), 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in sorted(OUTPUT_DIR.iterdir()):
        if fpath.name.endswith('.zip'):
            continue  # don't zip the zip
        zf.write(fpath, fpath.name)
print(f'Exported: {zip_path.name}')

# File listing
print(f'\nAll artifacts in {OUTPUT_DIR}:')
for fpath in sorted(OUTPUT_DIR.iterdir()):
    size_kb = fpath.stat().st_size / 1024
    print(f'  {fpath.name:50s} {size_kb:8.1f} KB')

In [ ]:
# ===========================================================================
# Cell: Download Buttons (Colab)
# ===========================================================================

# Download PDF of all figures
# Download ZIP of all artifacts
try:
    from google.colab import files
    
    print('=' * 50)
    print('  DOWNLOAD ARTIFACTS')
    print('=' * 50)
    print()
    print('Click below to download:')
    print()
    
    # Button 1: PDF
    print(f'[1] PDF: {pdf_path.name}')
    files.download(str(pdf_path))
    
    # Button 2: ZIP
    print(f'[2] ZIP: {zip_path.name}')
    files.download(str(zip_path))
    
except ImportError:
    print('Not running in Google Colab — skipping download buttons.')
    print(f'Artifacts are available in: {OUTPUT_DIR}')
    print(f'  PDF: {pdf_path}')
    print(f'  ZIP: {zip_path}')